# Uncertainty propagation

SmokeSight ships every measurement output with a documented uncertainty. This notebook shows where each sigma comes from and validates the analytic propagation against Monte Carlo.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass

from smokesight._atmos import IdentityAtmos
from smokesight._uncertainty import (
    radiance_uncertainty,
    tau_uncertainty,
    monte_carlo,
)

### 1. Radiance uncertainty

Per-pixel sigma_L combines four noise sources in quadrature:

    sigma_L = sqrt(shot^2 + read^2 + flatfield^2 + atmos^2)

- shot noise grows as sqrt(L)
- read noise (NER) is the floor at L=0
- flat-field uncertainty is a small relative term (~1%)
- atmospheric uncertainty depends on the model (5% fallback)

In [ ]:
@dataclass
class StubSensor:
    gain: float = 0.012
    noise_equivalent_radiance: float = 0.002
    flat_field_relative_uncertainty: float = 0.01

L_values = np.logspace(-3, 3, 100)
sigma_L = radiance_uncertainty(L_values, StubSensor(), IdentityAtmos())

fig, ax = plt.subplots(figsize=(7, 4))
ax.loglog(L_values, sigma_L, label='total sigma_L')
ax.loglog(L_values, np.sqrt(L_values * 0.012), '--', label='shot only')
ax.axhline(0.002, ls=':', label='read noise (NER)')
ax.set_xlabel('L [W m^-2 sr^-1 um^-1]')
ax.set_ylabel('sigma_L')
ax.set_title('Radiance uncertainty vs signal')
ax.legend()
plt.show()

At low signal the read-noise floor dominates; at high signal it's shot noise plus the 1% flat-field term.

### 2. Tau uncertainty (analytic vs Monte Carlo)

Beer-Lambert: tau = -ln(L / L0). First-order error propagation:

    sigma_tau = sqrt((sigma_L / L)^2 + (sigma_L0 / L0)^2)

For small relative errors this should match Monte Carlo.

In [ ]:
L0 = np.full(100, 1.0)
sigma_L0 = np.full(100, 0.05)
L = np.linspace(0.1, 1.0, 100)
sigma_L_arr = np.full(100, 0.05)

sigma_tau_analytic = tau_uncertainty(L, sigma_L_arr, L0, sigma_L0)

def beer_lambert(L_in, L0_in):
    return -np.log(L_in / L0_in)

mean_mc, sigma_tau_mc = monte_carlo(
    beer_lambert, [L, L0], [sigma_L_arr, sigma_L0], n=2000
)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(L, sigma_tau_analytic, label='analytic')
ax.plot(L, sigma_tau_mc, '--', label='Monte Carlo (n=2000)')
ax.set_xlabel('L / L0')
ax.set_ylabel('sigma_tau')
ax.set_title('Beer-Lambert uncertainty: analytic vs MC')
ax.legend()
plt.show()

The two curves overlap when (sigma / value) is small. Near L -> 0 the analytic formula correctly diverges (you can't measure optical depth through total absorption), while MC samples saturate.

### 3. Reproducibility

monte_carlo() is internally seeded with default_rng(42). Two identical calls return bit-identical arrays -- which is what makes SmokeSight uncertainties safe to compare across runs and CI workers.

In [ ]:
mean1, sig1 = monte_carlo(beer_lambert, [L, L0], [sigma_L_arr, sigma_L0], n=1000)
mean2, sig2 = monte_carlo(beer_lambert, [L, L0], [sigma_L_arr, sigma_L0], n=1000)
print(f'mean arrays identical:  {np.array_equal(mean1, mean2)}')
print(f'sigma arrays identical: {np.array_equal(sig1, sig2)}')

### 4. The masking contract

Pixels with invalid retrievals (low background confidence, ratio out of bounds, tau > tau_max, non-positive L0) come back as NaN in both tau and sigma_tau. A masked pixel never carries a numeric uncertainty -- that would be a measurement claim we can't back up. Downstream code filters on np.isfinite(tau) to skip masked cells.